# Homework 9 - Serverless



## Q1. Input / output node names

In [1]:
import onnxruntime as ort

sess = ort.InferenceSession('hair_classifier_v1.onnx', providers=['CPUExecutionProvider'])
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print('input :', inp.name, inp.shape)
print('output:', out.name, out.shape)

input : input ['s77', 3, 200, 200]
output: output ['s77', 1]


## Q2. Target size



In [2]:
import onnxruntime as ort
sess2 = ort.InferenceSession('hair_classifier_v1.onnx', providers=['CPUExecutionProvider'])
inp = sess2.get_inputs()[0]
print('Input shape:', inp.shape)
h, w = inp.shape[2], inp.shape[3]
print(f'Target size -> {(w, h)} (el mismo que en homework 8)')

Input shape: ['s77', 3, 200, 200]
Target size -> (200, 200) (el mismo que en homework 8)


## Q3. Pre-processing

In [3]:
import numpy as np
from PIL import Image

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

img = Image.open('test_img.jpeg')
img = prepare_image(img, (200, 200))

x = np.asarray(img, dtype=np.float32) / 255.0
x = (x - np.array([0.485, 0.456, 0.406], dtype=np.float32)) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
x = np.expand_dims(x.transpose(2, 0, 1), axis=0)

print('First pixel, R channel:', round(float(x[0, 0, 0, 0]), 3))

First pixel, R channel: -1.073


## Q4. Apply the model



In [4]:
pred = sess.run(None, {inp.name: x})[0][0][0]
print('Model output:', round(float(pred), 3))
print('If sigmoid is applied: ', round(float(1.0 / (1.0 + np.exp(-pred))), 3))

Model output: 0.092
If sigmoid is applied:  0.523


## Q5/Q6. Docker image `agrigorev/model-2025-hairstyle:v1` + Lambda



In [5]:
import subprocess
r = subprocess.run(['docker', 'images', 'agrigorev/model-2025-hairstyle:v1'],
                   capture_output=True, text=True)
print(r.stdout)
print('Nota: la imagen mide ~608 Mb con la columna SIZE de docker images en Linux.')

IMAGE                               ID             DISK USAGE   CONTENT SIZE   EXTRA
agrigorev/model-2025-hairstyle:v1   9e43d5a5323f        921MB          269MB        



Nota: la imagen mide ~608 Mb con la columna SIZE de docker images en Linux.


In [6]:
import subprocess, time
import requests

# Levanta el contenedor lambda si no esta corriendo (imagen hw9-lambda ya construida)
out = subprocess.run(['docker', 'ps', '-a', '--filter', 'name=hw9-test', '--format', '{{.Names}}'],
                     capture_output=True, text=True).stdout.strip()
if out:
    subprocess.run(['docker', 'rm', '-f', 'hw9-test'], capture_output=True)
subprocess.run(['docker', 'run', '-d', '--name', 'hw9-test', '-p', '8080:8080',
                'hw9-lambda', 'lambda.handler'], check=True, capture_output=True)
for _ in range(15):
    try:
        requests.get('http://localhost:8080/2015-03-31/functions/function/invocations', timeout=2)
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

resp = requests.post(
    'http://localhost:8080/2015-03-31/functions/function/invocations',
    json={'url': 'https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg'})
pred = resp.json()['prediction']
print('Q6 output del modelo:', round(float(pred), 3))

Q6 output del modelo: -0.102
